# The emergency network with `continuumx`

This notebook walks through `continuumx` on the weighted emergency network, following Chapter 5 of the master's thesis *Computing Distances in Combinatorial and Continuous Graphs* (A. Carrasco Martín, Universidad de Sevilla).

A town has seven road junctions, and each road is labeled with the time, in minutes, needed to traverse it. The town council must decide where to build a fire station so that the worst-case response time is as small as possible. In the discrete model, the station must be built at a junction; in the continuous model, it can be built anywhere along a road.

## 1. Building the continuous graph (Section 5.2)

The graph is built with NetworkX, storing the length of each road in the attribute `length`, and wrapped in a `ContinuousGraph`.

In [1]:
import networkx as nx
import continuumx as cx

roads = [
    ("a", "b", 1.0), ("b", "c", 3.0), ("c", "d", 1.0), ("d", "e", 1.0),
    ("e", "a", 1.0), ("a", "f", 2.0), ("c", "g", 2.0),
]

g = nx.Graph()
g.add_weighted_edges_from(roads, weight="length")

G = cx.ContinuousGraph(g)
G

<ContinuousGraph with 7 vertices, 7 edges, total length 11.0>

## 2. Points (Section 5.2)

A point is a triple `(u, v, θ)`: the edge `uv` and the position `θ ∈ [0, 1]` along it, measured from `u`. Every point is put in canonical form, so different descriptions of the same point compare equal.

In [2]:
cx.Point("b", "a", 0.25) == cx.Point("a", "b", 0.75)

True

In [3]:
cx.Point("a", "b", 0.0) == cx.Point("a", "c", 0.0)   # both are the vertex a

True

Points of a given graph are built with `G.point`, either from the parameter `θ` or from the distance to `u` (`offset`). The point `p` lies on the road `bc`, one minute away from `b`; since `ℓ(bc) = 3`, its parameter is `θ = 1/3`.

In [4]:
p = G.point("b", "c", offset=1.0)
p

Point('b', 'c', 0.3333333333333333)

In [5]:
G.offset(p)

1.0

In [6]:
q = G.point("d", "e", theta=0.25)
q

Point('d', 'e', 0.25)

## 3. Distances (Section 5.3)

The distance between two points uses only the distances between vertices, through the formulas of Proposition 4.1. The shortest route from `p` to `q` is `(p, c, d, q)`, of length 2 + 1 + 0.25.

In [7]:
cx.distance(G, p, q)

3.25

## 4. Eccentricity (Section 5.4)

The eccentricity of a point is its distance to the farthest point of the graph, which may lie inside an edge.

In [8]:
cx.eccentricity(G, G.point("d"))

4.0

In [9]:
cx.eccentricity(G, G.point("d", "e", theta=0.5))   # the midpoint of de

3.5

Along a whole edge, the eccentricity is a piecewise linear function of `θ`: the upper envelope of the hat functions of the edge. Along `de` it is `max{4 − θ, 3 + θ, 7/2}`, stored by its breakpoints.

In [10]:
f = cx.eccentricity_function(G, "d", "e")
f.points()

[(0.0, 4.0), (0.5, 3.5), (1.0, 4.0)]

## 5. Diameter (Section 5.5)

The diameter is the largest ceiling of the hat functions. Here it is attained by the pendant vertices `f` and `g`, and coincides with the discrete diameter.

In [11]:
cx.diameter(G)

7.0

In [12]:
cx.diametral_pair(G)

(Point('f'), Point('g'))

In [13]:
nx.diameter(g, weight="length")

7.0

## 6. Radius and center (Section 5.6)

This is where the two models differ. In the discrete model, the best a junction can do is a worst-case response time of 4 minutes, attained at `d` and `e`:

In [14]:
nx.radius(g, weight="length"), nx.center(g, weight="length")

(4.0, ['d', 'e'])

In the continuous model, the radius is 3.5 minutes, attained only at the midpoint of the road `de`, a point that is not a vertex:

In [15]:
cx.radius(G)

3.5

In [16]:
cx.center(G)

[Point('d', 'e', 0.5)]

The center is returned as a list of `Point` (isolated central points) and `Segment` (stretches of an edge made of central points). On a continuous cycle every point is central, so the center is the whole graph:

In [17]:
cx.center(nx.cycle_graph(4))

[Segment(0, 1, 0.0, 1.0), Segment(0, 3, 0.0, 1.0), Segment(1, 2, 0.0, 1.0), Segment(2, 3, 0.0, 1.0)]

## 7. Exact arithmetic

Lengths may also be given as `fractions.Fraction`. The algorithms use only the four arithmetic operations and comparisons, so with rational lengths every result is an exact rational. For instance, the parameter of `p` is now exactly `1/3`:

In [18]:
from fractions import Fraction

h = nx.Graph()
h.add_weighted_edges_from(
    [(u, v, Fraction(ell)) for u, v, ell in roads], weight="length"
)
H = cx.ContinuousGraph(h, tol=0)

H.point("b", "c", offset=Fraction(1))

Point('b', 'c', Fraction(1, 3))

In [19]:
cx.radius(H), cx.center(H)

(Fraction(7, 2), [Point('d', 'e', Fraction(1, 2))])